In [1]:
from os.path import join, dirname
import os
import sys
from copy import deepcopy
from hydra import initialize, compose
from omegaconf import DictConfig
import numpy as np
import torchshow as ts
import torchvision

import torch
from torch import autocast
from torch.cuda.amp import GradScaler
from torch.nn import DataParallel
import torch.nn.functional as F
from einops import rearrange

sys.path.append(join(dirname(dirname(os.getcwd()))))
sys.path.append(os.getcwd())

from omniisaacgymenvs.utils.hydra_cfg.hydra_utils import *
from omniisaacgymenvs.utils.hydra_cfg.reformat import omegaconf_to_dict, print_dict
from omniisaacgymenvs.utils.task_util import initialize_task
from omniisaacgymenvs.envs.vec_env_rlgames import VecEnvRLGames

from memoryPerceiver.memory_perceiver.main.configuration import cf
from memoryPerceiver.memory_perceiver.core.model_wrapper import create_memories, reset_memories
import memoryPerceiver.memory_perceiver.isaac.trainer as tr
from memoryPerceiver.memory_perceiver.isaac.plot import Plotter
from memoryPerceiver.memory_perceiver.isaac.agent import ActorCritic, ActorCriticSimple, ActorCriticPerceiver

%load_ext autoreload
%autoreload 2
%env ISAAC_JUPYTER_KERNEL=1

env: ISAAC_JUPYTER_KERNEL=1


/isaac-sim/kit/python/lib/python3.7/site-packages/IPython/extensions/autoreload.py:121: DeprecationWarning: the imp module is deprecated in favour of importlib; see the module's documentation for alternative uses
  from imp import reload


In [2]:
num_gpus=torch.cuda.device_count()
device_cpu = torch.device("cpu")
device = torch.device(0 if torch.cuda.is_available() else "cpu")
device_agents = torch.device(1 if num_gpus>1 else 0)
devices_agents = [torch.device(i) for i in range(min(1, num_gpus-1), num_gpus)]
torch.use_deterministic_algorithms(False)
torch.backends.cudnn.benchmark = True
torch.backends.cudnn.deterministic = False
%env CUBLAS_WORKSPACE_CONFIG=:4096:8

env: CUBLAS_WORKSPACE_CONFIG=:4096:8


In [3]:
"""model = ActorCriticSimple(cf, device)
model_student = ActorCriticSimple(cf, device)

#actions, logs, vals, dist = model.act(torch.randn(8,4, device=device))
#print(actions.size(), logs.size(), vals.size())
"""

'model = ActorCriticSimple(cf, device)\nmodel_student = ActorCriticSimple(cf, device)\n\n#actions, logs, vals, dist = model.act(torch.randn(8,4, device=device))\n#print(actions.size(), logs.size(), vals.size())\n'

In [4]:
model = DataParallel(ActorCriticPerceiver(cf, device), device_ids=devices_agents)
model_student = DataParallel(ActorCriticPerceiver(cf, device), device_ids=devices_agents)
"""
num_envs=1
states = {'observations' : torch.randn(num_envs,cf.frames,4, device=device),
         'video' : torch.zeros(num_envs, 3, cf.frames, 256, 456, device=device),
         'audio':torch.zeros(num_envs, cf.frames*1920, device=device)
        }
model.act(states)
"""

"\nnum_envs=1\nstates = {'observations' : torch.randn(num_envs,cf.frames,4, device=device),\n         'video' : torch.zeros(num_envs, 3, cf.frames, 256, 456, device=device),\n         'audio':torch.zeros(num_envs, cf.frames*1920, device=device)\n        }\nmodel.act(states)\n"

In [5]:
#@hydra.main(config_name="config", config_path="../cfg")
#@hydra.main(config_name="config", config_path="../../OmniIsaacGymEnvs/omniisaacgymenvs/cfg")
def isaac_training(cfg: DictConfig):
    cfg_dict = omegaconf_to_dict(cfg)
    env = VecEnvRLGames(headless=cfg.headless, sim_device=cfg.device_id, enable_viewport=cfg.enable_viewport)
    
    
    # !!! Load isaac modules after the variables
    from omni.isaac.core.utils.torch.maths import set_seed
    import omni.usd
    from omni.isaac.core.utils.extensions import enable_extension
    enable_extension("omni.isaac.sensor")
    from omni.isaac.core.utils.stage import open_stage, save_stage
    
    plotter = Plotter(titles=['loss_actor', 'loss_critic', 'rewards'])
    history_keys = ['actions', 'values', 'logprobs', 'returns', 'rewards', 'advantages', 'death_index', 'generator']
    history = tr.init_history(history_keys, sim_duration, num_envs, cf.dim_actions_ppo, device)
    previous_history = deepcopy(history)
    buffer = {'video':[], 'audio':[], 'obs':[], 'rewards':[], 'actions_star':[]}
    memories = create_memories(model.module.actor, cf.batch_size, device=device)
    last_memories = torch.zeros_like(memories['cross_cmem'], device=device)

    cfg.seed = set_seed(cfg.seed, torch_deterministic=cfg.torch_deterministic)
    cfg_dict['seed'] = cfg.seed  # sets seed. if seed is -1 will pick a random one
    task = initialize_task(cfg_dict, env)
    all_losses_actor, all_losses_critic, all_returns = [], [], []
    optimizer = torch.optim.Adam(model_student.parameters(), lr=learning_rate)
    scaler = GradScaler()
    sim_count, sim_step = -1, 0
    is_inference=False  # Will switch to True immediately
    
    while env._simulation_app.is_running():
        if env._world.is_playing():
            # if env._world.current_time_step_index == 0 or \
            # env._world.current_time_step_index % sim_duration == 0:
            if env.sim_frame_count % sim_duration == 0:
                
                ### Plots

                if sim_count % 6 == 1 and len(all_losses_actor)>0:
                    plotter.plot_all([(all_losses_actor, 160), (all_losses_critic, 160), all_returns])

                ### Initialization

                sim_count+=1
                env._world.reset(soft=False)
                is_inference = not is_inference
                
                print('$$$$$$$$$$$ CUMUL REWARD $$$$$$$$$$$')
                if len(history['rewards'])>0:
                    print(history['rewards'].sum())
                if is_inference:
                    print("MEM: ", F.mse_loss(last_memories, memories['cross_cmem']).item())
                    print('PLAY')
                    if sim_count == 0 and os.path.exists(path_model):
                        model.module.load_state_dict(torch.load(path_model))
                        model_student.module.load_state_dict(torch.load(path_model))
                    else:
                        if sim_count % student_update_step*2 == 0:
                            model.module.load_state_dict(model_student.module.state_dict())
                    torch.save(model_student.module.state_dict(), path_model)
                    model.eval()
                else:
                    print('REPLAY')
                    last_memories = memories['cross_cmem'].clone()
                    model_student.train() 
                    optimizer.zero_grad()
                    previous_history = deepcopy(history)
                    previous_history['returns'] = tr.calculate_returns(history['rewards'], history['death_index'], gamma=c_gamma)
                    previous_history['advantages'] = tr.calculate_gae(history['rewards'], history['values'], 
                                                        history['death_index'], gamma=c_gamma, gae_lambda=c_lambda)
                    all_returns.append(previous_history['returns'][1, :].mean().item())
                    
                
                tr.set_seeds(sim_count//2)
                print("SEED: ", sim_count//2)
                reset_memories(memories)
                full_video = []
                birth_time = torch.zeros(num_envs)
                for key in buffer.keys():
                    buffer[key]=[]
                history = tr.init_history(history_keys, sim_duration, num_envs, cf.dim_actions_ppo, device)
                backward_count, kl_div = 0, 0
                sim_step = 0

            ### Get agent observations 

            all_obs = env._task.get_observations()['articulation_view']
            obs = all_obs['obs_buf'].clone()    # (num_envs, d)
            video = all_obs['video_buf'].clone()  # (num_envs, 256, 456, 3)
            #ts.save(video[0], '/isaac-sim/memoryPerceiver/memory_perceiver/main/monitor/live.jpg')
            full_video.append(video.cpu())
            env._task.calculate_metrics()       
            reward=env._task.rew_buf.clone()    # (num_envs, 1)
            tr.update_buffers(buffer, {'obs': obs, 'video': video, 'rewards': reward}, frames=cf.frames)
            history['rewards'][sim_step] = reward

            ### Reset memory at death and record 

            reset_indexes = []
            for agent_i in range(num_envs):
                if env._task.reset_buf[agent_i].item() == 1:
                    reset_indexes.append(agent_i)
                    birth_time[agent_i] = sim_step

                    # Update death dictionnary during inference

                    if is_inference:
                        history['death_index'][sim_step, agent_i]=1
            reset_memories(memories, indexes=reset_indexes)

            ### Agent
            
            if len(buffer['actions_star'])==0 :

                ## Process inputs

                states={}
                mask = tr.create_mask(birth_time, sim_step, frames=cf.frames)
                states['observations'] = tr.buffer_padding(torch.stack(buffer['obs']), [cf.dim_observations],
                                                            num_envs=num_envs, frames=cf.frames)
                states['observations'][mask] *= 0
                states['video'] = torch.zeros(num_envs, cf.frames, cf.height, cf.width, 3, device=device)
                # states['video'] = tr.buffer_padding(torch.stack(buffer['video']), [cf.height, cf.width, 3], num_envs=num_envs, frames=cf.frames)
                states['video'][mask] *= 0
                states['video'] = rearrange(states['video'], 'b t h w c -> b c t h w')
                states['audio'] = torch.zeros(num_envs, cf.frames*1920, device=device)

                ## Inference, save history and memory update

                if is_inference:
                    for i_gpu in range(num_gpus):
                        history['generator'][sim_step, i_gpu] = torch.cuda.get_rng_state(i_gpu).clone() 
                    with autocast(device_type='cuda', dtype=torch.float16):
                        actions, logprobs, values, memories = model(states, memories)   
                    history['actions'][sim_step] = actions  
                    history['logprobs'][sim_step] = logprobs 
                    history['values'][sim_step] = values     
                else:
                    actions, logprobs, values = [previous_history[key][sim_step].to(device_agents) \
                                                 for key in ['actions', 'logprobs', 'values']]
                                    

                ## Backward pass

                if not is_inference:

                    with autocast(device_type='cuda', dtype=torch.float16):
                        returns = previous_history['returns'][sim_step].to(device_agents)
                        advantages = previous_history['advantages'][sim_step].to(device_agents)
                        new_logprobs, new_values, entropies, _ = model_student(states, memories, actions)
                        for i_gpu in range(num_gpus):
                            torch.cuda.set_rng_state(previous_history['generator'][sim_step, i_gpu], device=i_gpu)
                        _, _, _, memories = model(states, memories) 
                        

                        # Loss
                        ratios = torch.exp(new_logprobs - logprobs)
                        surr1 = ratios * advantages
                        surr2 = torch.clamp(ratios, 1-eps_clip, 1+eps_clip) * advantages
                        loss_actor = -(torch.min(surr1, surr2)).mean()
                        loss_entropy = -c_entropy * entropies.mean()
                        loss_critic = c_critic * F.mse_loss(new_values, returns)
                        loss =  (loss_actor + loss_critic + loss_entropy) /accumulation_steps
                    scaler.scale(loss).backward()
                    backward_count += 1
                    #kl_div += torch.distributions.kl_divergence(new_dist, old_dist).mean()/accumulation_steps


                    # Step optimizer

                    if backward_count % accumulation_steps == 0 and backward_count != 0:
                        scaler.unscale_(optimizer)
                        torch.nn.utils.clip_grad_norm_(model_student.parameters(), c_grad_norm)
                        scaler.step(optimizer)
                        scaler.update()
                        optimizer.zero_grad()

                        # Adaptive schedule
                        #for param_group in optimizer.param_groups:
                        #    if adaptive_schedule:
                        #        param_group['lr'] = tr.update_lr(param_group['lr'], kl_div)

                        # Reinitialize
                        backward_count = 0
                        # kl_div = 0


                    all_losses_actor.append(loss_actor.item())
                    all_losses_critic.append(loss_critic.item())

                # Update buffers

                actions_star = model.module.instanciate_actions(actions, dt=cf.frames_future)
                buffer['actions_star'] += list(torch.unbind( actions_star ))


            ## Current action (num_envs, 1)

            if(len(buffer['actions_star'])>0):
                action = buffer['actions_star'].pop(0)
                action = torch.clamp(action, min=-1, max=1)
            else:
                action = torch.dzeros(num_envs, 1)

            if sim_count % 10 == 0 and env.sim_frame_count % sim_duration == 256:
                torchvision.io.write_video('/isaac-sim/memoryPerceiver/memory_perceiver/main/monitor/test_'  + str(sim_count) + '_vid_.mp4', 
                                            torch.stack(full_video)[:,1], 8*cf.frames)

            # Step environment

            if sim_step == 0:
                env._task.post_reset()
            env._task.pre_physics_step(action.to(task.rl_device))
            env._world.step(render=render)
            env.sim_frame_count += 1
            env._task.post_physics_step()

            sim_step+=1

        else:
            env._world.step(render=render)

    env._simulation_app.close()


if __name__ == '__main__':
    sim_duration = 260
    num_envs = cf.batch_size
    learning_rate = 1e-4
    accumulation_steps = 4
    c_entropy = 0.05
    c_critic=0.2
    student_update_step = 1
    eps=1e-9
    c_gamma=0.99
    c_lambda=0.95
    c_grad_norm=0.1
    eps_clip=0.2
    adaptive_schedule=False
    render = True
    path_model = 'weights/humanoid.pt'
    with initialize(config_path="memoryPerceiver/OmniIsaacGymEnvs/omniisaacgymenvs/cfg"):
        cfg = compose(config_name="config", overrides=["task=Humanoid", "headless=True","enable_viewport=True",
                        "num_envs="+str(num_envs), "imgWidth="+str(cf.width), "imgHeight="+str(cf.height)])
    #print(OmegaConf.to_yaml(cfg))
    isaac_training(cfg)

/isaac-sim/kit/python/lib/python3.7/site-packages/hydra/_internal/defaults_list.py:251: UserWarning: In 'config': Defaults list is missing `_self_`. See https://hydra.cc/docs/upgrades/1.0_to_1.1/default_composition_order for more information
  warnings.warn(msg, UserWarning)
/isaac-sim/kit/python/lib/python3.7/site-packages/hydra/_internal/defaults_list.py:412: UserWarning: In config: Invalid overriding of hydra/job_logging:
Default list overrides requires 'override' keyword.
See https://hydra.cc/docs/next/upgrades/1.0_to_1.1/defaults_list_override for more information.

  deprecation_warning(msg)


Starting kit application with the following args:  ['/isaac-sim/exts/omni.isaac.kit/omni/isaac/kit/simulation_app.py', '/isaac-sim/apps/omni.isaac.sim.python.gym.headless.render.kit', '--/app/tokens/exe-path=/isaac-sim/kit', '--/persistent/app/viewport/displayOptions=3094', '--/rtx/materialDb/syncLoads=True', '--/rtx/hydra/materialSyncLoads=True--/omni.kit.plugin/syncUsdLoads=True', '--/app/renderer/resolution/width=1280', '--/app/renderer/resolution/height=720', '--/app/window/width=1440', '--/app/window/height=900', '--/renderer/multiGpu/enabled=True', '--/app/fastShutdown=False', '--ext-folder', '/isaac-sim/exts', '--ext-folder', '/isaac-sim/apps', '--/physics/cudaDevice=0', '--portable', '--no-window', '--allow-root']
Passing the following args to the base kit application:  ['-f', '/root/.local/share/jupyter/runtime/kernel-e23861f1-56b0-488a-b28c-d752bff4f43f.json']
[Warning] [omni.isaac.kit.simulation_app] Modules: ['omni.isaac.kit.app_framework'] were loaded before SimulationApp 

2024-03-30 23:31:59 s] [Error] [carb.glinterop.plugin] GLInteropContext::init: carb::windowing is not available



|---------------------------------------------------------------------------------------------|
| Driver Version: 535.161.07    | Graphics API: Vulkan
|=============================================================================================|
| GPU | Name                             | Active | LDA | GPU Memory | Vendor-ID | LUID       |
|     |                                  |        |     |            | Device-ID | UUID       |
|---------------------------------------------------------------------------------------------|
| 0   | NVIDIA GeForce RTX 3060          | Yes: 0 |     | 12288   MB | 10de      | 0          |
|     |                                  |        |     |            | 2487      | 4f2753eb.. |
|---------------------------------------------------------------------------------------------|
| 1   | NVIDIA GeForce RTX 3060          | Yes: 1 |     | 12288   MB | 10de      | 0          |
|     |                                  |        |     |            | 2504     

/isaac-sim/kit/kernel/py/omni/kit/app/_impl/__init__.py:104: DeprecationWarning: Using or importing the ABCs from 'collections' instead of from 'collections.abc' is deprecated since Python 3.3,and in 3.9 it will stop working
  if not isinstance(setting_py, str) and isinstance(setting_py, collections.Sequence):


...

/isaac-sim/kit/exts/omni.graph/omni/graph/core/_impl/autonode/type_definitions.py:12: DeprecationWarning: `np.float` is a deprecated alias for the builtin `float`. To silence this warning, use `float` by itself. Doing this will not modify any behavior and is safe. If you specifically wanted the numpy scalar type, use `np.float64` here.
Deprecated in NumPy 1.20; for more details and guidance: https://numpy.org/devdocs/release/1.20.0-notes.html#deprecations
  Float = NewType("Float", np.float)


Warp 0.6.3 initialized:
   CUDA Toolkit: 11.5, Driver: 12.2
   Devices:
     "cpu"    | x86_64
     "cuda:0" | NVIDIA GeForce RTX 3060 (sm_86)
     "cuda:1" | NVIDIA GeForce RTX 3060 (sm_86)
   Kernel cache: /root/.cache/warp/0.6.3
 omni.mdl-0.1.0] startup
[1.688s] [ext: omni.kit.viewport.registry-104.0.2] startup
[1.689s] [ext: omni.kit.menu.create-1.0.8] startup
[1.690s] [ext: omni.kit.material.library-1.3.21] startup
[1.696s] [ext: omni.kit.viewport.legacy_gizmos-1.0.10] startup
[1.700s] [ext: omni.kit.hydra_texture-1.0.11] startup
[1.705s] [ext: omni.kit.context_menu-1.5.12] startup
[1.708s] [ext: omni.ui.scene-1.5.18] startup
[1.712s] [ext: omni.kit.window.drop_support-1.0.1] startup
[1.713s] [ext: omni.kit.widget.viewport-104.1.16] startup
[1.715s] [ext: omni.kit.window.preferences-1.3.8] startup
[1.779s] [ext: omni.usdphysics-104.2.4-5.1] startup
[1.783s] [ext: omni.kit.viewport.window-104.1.23] startup
2024-03-30 23:32:00 [1,779ms] [Warning] [omni.kit.ui] Failed to set value fo

/isaac-sim/kit/python/lib/python3.7/site-packages/gym/spaces/box.py:84: UserWarning: WARN: Box bound precision lowered by casting to float32
  logger.warn(f"Box bound precision lowered by casting to {self.dtype}")


[6.251s] app ready
[6.331s] Simulation App Startup Complete
[6.343s] [ext: omni.usd.schema.isaac-1.1.0] startup
[6.370s] [ext: omni.inspect-1.0.1] startup
[6.373s] [ext: omni.graph.action-1.31.1] startup
[6.382s] [ext: omni.graph.visualization.nodes-2.0.0] startup
[6.390s] [ext: omni.kit.widget.graph-1.5.6-104_2] startup
[6.400s] [ext: omni.kit.window.property-1.8.2] startup
[6.403s] [ext: omni.kit.window.extensions-1.1.1] startup
[6.407s] [ext: omni.syntheticdata-0.2.4] startup
2024-03-30 23:32:05 [6,413ms] [Warning] [omni.kit.viewport.menubar.core.menu_item.viewport_menu_item] ViewportMenuItem order argument is deprecated, use order_setting_path
2024-03-30 23:32:05 [6,413ms] [Warning] [omni.syntheticdata.scripts.extension] SyntheticData extension needs at least a stageFrameHistoryCount of 3
[6.423s] [ext: omni.isaac.ui-0.8.0] startup
[6.426s] [ext: omni.debugdraw-0.1.1] startup
[6.432s] [ext: usdrt.scenegraph-4.3.2] startup
[6.448s] [ext: omni.isaac.debug_draw-0.2.3] startup
[6.453s]

2024-03-31 00:03:19 [1,881,052ms] [Warning] [carb] Plugin interface for a client: omni.hydratexture.plugin was already released.
2024-03-31 00:03:19 [1,881,052ms] [Warning] [carb] Plugin interface for a client: omni.hydratexture.plugin was already released.
2024-03-31 00:03:19 [1,881,052ms] [Warning] [carb] Plugin interface for a client: omni.hydratexture.plugin was already released.
2024-03-31 00:03:19 [1,881,052ms] [Warning] [carb] Plugin interface for a client: omni.hydratexture.plugin was already released.
2024-03-31 00:03:19 [1,881,052ms] [Warning] [carb] Plugin interface for a client: omni.hydratexture.plugin was already released.
2024-03-31 00:03:19 [1,881,052ms] [Warning] [carb] Plugin interface for a client: omni.hydratexture.plugin was already released.
2024-03-31 00:03:19 [1,881,052ms] [Warning] [carb] Plugin interface for a client: omni.hydratexture.plugin was already released.
2024-03-31 00:03:19 [1,881,052ms] [Warning] [carb] Plugin interface for a client: omni.hydratextu

In [ ]:
if __name__ == '__main__':
    with initialize(config_path="memoryPerceiver/OmniIsaacGymEnvs/omniisaacgymenvs/cfg"):
        cfg = compose(config_name="config", overrides=["task=Humanoid", "headless=True","enable_viewport=True",
                        "num_envs="+str(7)])
    print(cfg.num_envs)
    print(OmegaConf.to_yaml(cfg))

In [ ]:
!nvidia-smi

: 